## Conditional Chain - Custom runnable

In [133]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableBranch
from pydantic import BaseModel
from typing import Literal
from langchain_core.runnables import RunnablePassthrough



# 1. Explicitly load the variables from your .env file
load_dotenv()

# 2. (Optional) Run a quick sanity print statement to check it loaded
print("Loaded Key prefix:", os.environ.get("OPENAI_API_KEY")[:6])

# 3. Initialize your LangChain architecture
llm_openai = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0)

Loaded Key prefix: sk-pro


In [134]:
class llm_schema(BaseModel):

    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

result = llm_structured_output.invoke("Film was okayish")

result.model_dump()['movie_summary_flag']

# demo_chain = llm_structured_output | PydanticOutputParser(pydantic_object = llm_schema)

# demo_chain.invoke("Film was okayish")

'negative'

In [135]:
#Task 1

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie reviewer "),
        ("user",  "Please tell the review of the movie is positive or negative: {input}")
    ]
)

In [136]:
# Task 2 [LLM]

llm_structured_output = llm_openai.with_structured_output(llm_schema)


In [137]:
#3 Custom task for parsing the text to positive or negative alone
def extract_sentiment(input_data:llm_schema)->str:
        return input_data.movie_summary_flag 

extract_sentiment_runnable = RunnableLambda(extract_sentiment)

In [138]:
sentiment_chain = prompt_template | llm_structured_output | extract_sentiment_runnable


### Conditional Chain 1 - Positive

In [139]:
#Task 1 - Chain 1

linkedin_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a linkedin post generator. Return Gold Star if it is positive"),
        ("user",  "Create a post for the following text : {text}")
    ]
)

#Task 2 - LLM

llm_openai = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0)

# Task 3 - String parser

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser


## Conditional Chain 2 - Negative

In [140]:
## Conditional Chain 2 - Negative
insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an Instagram post generator."),
    ("user", "Create a post for the following text : {text}")
])
chain_insta = insta_prompt | llm_openai | StrOutputParser()


## Final Orchestration

In [141]:

# This function decides which chain to run based on the sentiment data
def route_inputs(inputs: dict):
    sentiment = inputs["sentiment"]
    text_content = inputs["original_input"]
    
    if sentiment == "positive":
        return chain_linkedin.invoke({"text": text_content})
    else:
        return chain_insta.invoke({"text": text_content})

router_runnable = RunnableLambda(route_inputs)

## Final Orchestration
final_orchestration = (
    RunnablePassthrough.assign(sentiment=sentiment_chain)
    | RunnablePassthrough.assign(original_input=lambda x: x["input"])
    | router_runnable
)

# Execute the complete pipeline
output = final_orchestration.invoke({"input": "I like it"})
print(output)


Gold Star


In [142]:
final_orchestration.invoke({"input":"I  like it"})

'Gold Star'